In [ ]:
import logging
logging.basicConfig(
	filename='app.log',
	level=logging.INFO,
	filemode='w',  # 'w' for write (overwrite), 'a' for append (default)
	format='%(asctime)s - %(levelname)s - %(message)s'
)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from core.CNNmodel import *
from core.Log import *
from core.preprocessing import *
from core.plots import *
from core.mydataloader import *

CNTRL_dicom_root = "../Takotsubo-Syndrome/data/Inputs/normal_cases/"
TTS_dicom_root = "../Takotsubo-Syndrome/data/Inputs/takotsubo_cases/"
root_dir = "data/cases/"



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn as nn
import numpy as np
import copy
from sklearn.metrics import roc_auc_score
import logging

def _train_epoch(model, loader, optimizer, criterion, device):
	"""Helper function for a single training epoch."""
	model.train()
	running_loss = 0.0
	all_labels = []
	all_preds = []

	for batch in loader:
		# Use the correct keys from your TTSDataset
		axial = batch["axial_image"].to(device)
		coronal = batch["coronal_image"].to(device)
		sagittal = batch["sagittal_image"].to(device)
		clinical = batch["clinical_features"].to(device)
		labels = batch["label"].to(device).unsqueeze(1)

		optimizer.zero_grad()
		outputs = model(axial, sagittal, coronal, meta=clinical)
		loss = criterion(outputs, labels)
		loss.backward()
		optimizer.step()

		running_loss += loss.item() * labels.size(0)

		# Store preds and labels for accuracy calculation
		preds = torch.sigmoid(outputs) > 0.5
		all_preds.extend(preds.cpu().numpy())
		all_labels.extend(labels.cpu().numpy())

	epoch_loss = running_loss / len(loader.dataset)
	epoch_acc = np.mean(np.array(all_preds) == np.array(all_labels))
	return epoch_loss, epoch_acc

def _validate_epoch(model, loader, criterion, device):
	"""Helper function for a single validation epoch."""
	model.eval()
	running_loss = 0.0
	all_labels = []
	all_preds = []

	with torch.no_grad():
		for batch in loader:
			axial = batch["axial_image"].to(device)
			coronal = batch["coronal_image"].to(device)
			sagittal = batch["sagittal_image"].to(device)
			clinical = batch["clinical_features"].to(device)
			labels = batch["label"].to(device).unsqueeze(1)

			outputs = model(axial, sagittal, coronal, meta=clinical)
			loss = criterion(outputs, labels)

			running_loss += loss.item() * labels.size(0)
			preds = torch.sigmoid(outputs) > 0.5
			all_preds.extend(preds.cpu().numpy())
			all_labels.extend(labels.cpu().numpy())

	epoch_loss = running_loss / len(loader.dataset)
	epoch_acc = np.mean(np.array(all_preds) == np.array(all_labels))
	return epoch_loss, epoch_acc

def train_model(model, train_loader, val_loader, hypers):
	"""
	Trains the model and returns the best version based on validation loss.

	Args:
		model (nn.Module): The PyTorch model to train.
		train_loader (DataLoader): The data loader for training.
		val_loader (DataLoader): The data loader for validation.
		hypers (dict): A dictionary of hyperparameters (LR, WD, epochs, etc.).

	Returns:
		nn.Module: The model with the best weights loaded.
	"""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)

	# Calculate pos_weight for handling class imbalance
	# This is a robust way to do it from the dataset itself
	labels = [d['label'] for d in train_loader.dataset.data_dicts]
	pos_weight = torch.tensor(labels.count(0) / labels.count(1), dtype=torch.float32)

	optimizer = optim.Adam(model.parameters(), lr=hypers['LR'], weight_decay=hypers['WD'])
	scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=hypers['patience'], factor=0.5, verbose=False)
	criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

	best_val_loss = float('inf')
	best_model_state = None
	patience_counter = 0

	logging.info("Starting model training...")
	for epoch in range(hypers['epochs']):
		train_loss, train_acc = _train_epoch(model, train_loader, optimizer, criterion, device)
		val_loss, val_acc = _validate_epoch(model, val_loader, criterion, device)

		logging.info(f"Epoch {epoch+1}/{hypers['epochs']} | "
					 f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
					 f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

		scheduler.step(val_loss)

		if val_loss < best_val_loss:
			best_val_loss = val_loss
			# Use deepcopy to save the state in memory, not to disk
			best_model_state = copy.deepcopy(model.state_dict())
			patience_counter = 0
		else:
			patience_counter += 1
			if patience_counter >= hypers['patience']:
				logging.info("Early stopping triggered.")
				break

	# Load the best model weights before returning
	model.load_state_dict(best_model_state)
	return model


In [ ]:
def evaluate_model(model, test_loader, hypers):
	"""
	Evaluates the final model on the test set.

	Returns:
		dict: A dictionary containing performance metrics (loss, accuracy, AUC).
	"""
	device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
	model.to(device)
	model.eval()

	# We need the same pos_weight for consistent loss calculation
	labels = [d['label'] for d in test_loader.dataset.data_dicts]
	pos_weight = torch.tensor(labels.count(0) / labels.count(1), dtype=torch.float32)
	criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

	running_loss = 0.0
	y_true = []
	y_prob = []

	with torch.no_grad():
		for batch in test_loader:
			axial = batch["axial_image"].to(device)
			coronal = batch["coronal_image"].to(device)
			sagittal = batch["sagittal_image"].to(device)
			clinical = batch["clinical_features"].to(device)
			labels = batch["label"].to(device).unsqueeze(1)

			outputs = model(axial, sagittal, coronal, meta=clinical)
			loss = criterion(outputs, labels)
			running_loss += loss.item() * labels.size(0)

			# Store probabilities for AUC and true labels
			probs = torch.sigmoid(outputs)
			y_prob.extend(probs.cpu().numpy())
			y_true.extend(labels.cpu().numpy())

	y_true = np.array(y_true)
	y_prob = np.array(y_prob)
	y_pred = (y_prob > hypers['threshold_cutoff']).astype(int)

	final_loss = running_loss / len(test_loader.dataset)
	final_acc = np.mean(y_pred == y_true)
	final_auc = roc_auc_score(y_true, y_prob) # Calculate AUC

	logging.info(f"Test Results -> Loss: {final_loss:.4f}, Accuracy: {final_acc:.4f}, AUC: {final_auc:.4f}")

	return {"loss": final_loss, "accuracy": final_acc, "auc": final_auc}


In [ ]:

def run_one_fold(train_datalist, val_datalist, test_datalist, fold_idx):
	"""
	Executes the entire pipeline for a single fold of the cross-validation.
	This includes:
	1. Calculating normalization stats for the fold's training data.
	2. Creating DataLoaders.
	3. Initializing and training the model.
	4. Evaluating the model on the fold's test set.
	Args:
		train_datalist (list): The list of training cases for this fold.
		val_datalist (list): The list of validation cases for this fold.
		test_datalist (list): The list of test cases for this fold.
		fold_idx (int): The index of the current fold (for logging).
	Returns:
		float: The performance score (e.g., AUC) for this fold.
	"""
	# --- Define Hyperparameters ---
	hypers = {
		"LR": 1e-4, "WD": 1e-5, "epochs": 50,
		"patience": 10, "batch_size": 8, "threshold_cutoff": 0.5
	}

	logging.info(f"--- Starting Fold {fold_idx + 1} ---")
	logging.info(f"Fold Split: {len(train_datalist)} train, {len(val_datalist)} val, {len(test_datalist)} test.")


	HUstats = [case["stats"] for case in train_datalist]
	UH_mean, HU_std = calculate_HU_stats(HUstats)

	ages = [case['age'] for case in train_datalist]
	AGE_mean = np.mean(ages); AGE_std = np.std(ages)
	logging.info(f"Fold {fold_idx + 1} | HU_mean={UH_mean:.2f}, HU_std={HU_std:.2f}, AGE_mean={AGE_mean:.2f}, AGE_std={AGE_std:.2f}")
	fold_stats = {
		'HU_mean': UH_mean,
		'HU_std': HU_std,
		'AGE_mean': AGE_mean,
		'AGE_std': AGE_std}

	train_loader, val_loader, test_loader = get_data_loaders(
		train_datalist, val_datalist, test_datalist, fold_stats)

	# 3. Initialize model and train it
	# This function would contain your epoch loop, training, validation, and saving the best model
	model = MultiViewCNN(input_size=(96, 96),
		use_metadata=True, kernel_size=3, padding=1, dropout_rate=0.5)
	best_model = train_model(model, train_loader, val_loader, hypers)

	# 4. Evaluate the final model on the held-out test set
	final_scores = evaluate_model(best_model, test_loader, hypers)

	logging.info(f"--- Fold {fold_idx + 1} Score: {final_scores['auc']:.4f} ---")

	return final_scores['auc']




In [ ]:
def main_training_pipeline():

	# --- 1. Load the datalist ---
	json_path = Path('data/data_info.json')
	full_datalist = load_dataset_info(json_path)
	if not full_datalist:
		logging.error("Datalist not loaded.")
		return

	# Prepare indices and labels for stratified splitting
	indices = np.arange(len(full_datalist))
	labels = [d['label'] for d in full_datalist]

	# --- 2. Setup the Outer Cross-Validation Loop ---
	N_OUTER_SPLITS = 5
	outer_cv = StratifiedKFold(n_splits=N_OUTER_SPLITS, shuffle=True, random_state=42)

	fold_scores = []

	logging.info(f"Starting {N_OUTER_SPLITS}-fold Nested Cross-Validation...")

	# The loop that creates the 5 different test sets
	for fold_idx, (train_val_idx, test_idx) in enumerate(outer_cv.split(indices, labels)):

		# --- 3. Outer Split: (Train+Val Pool) vs. Test Set ---
		# This creates the held-out test set for the current fold
		test_datalist_fold = [full_datalist[i] for i in test_idx]

		# This creates the pool of data that will be used for training and validation
		train_val_datalist_fold = [full_datalist[i] for i in train_val_idx]
		train_val_labels_fold = [d['label'] for d in train_val_datalist_fold]

		# --- 4. Inner Split: Train Set vs. Validation Set ---
		# Now we split the pool from the step above.
		# test_size=0.25 on the remaining 80% gives an overall 60% train / 20% val / 20% test split.
		inner_sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
		train_idx, val_idx = next(inner_sss.split(np.arange(len(train_val_datalist_fold)), train_val_labels_fold))

		# Create the final datalists for this fold
		train_datalist_fold = [train_val_datalist_fold[i] for i in train_idx]
		val_datalist_fold = [train_val_datalist_fold[i] for i in val_idx]

		# --- 5. Run the Training and Evaluation for this Fold ---
		# Pass the three distinct datasets to your core function
		score = run_one_fold(
			train_datalist=train_datalist_fold,
			val_datalist=val_datalist_fold,
			test_datalist=test_datalist_fold,
			fold_idx=fold_idx
		)
		fold_scores.append(score)

	# --- 6. Report Final Results ---
	mean_score = np.mean(fold_scores)
	std_score = np.std(fold_scores)

	logging.info("--- Nested Cross-Validation Complete ---")
	logging.info(f"Final Scores across {N_OUTER_SPLITS} folds: {[f'{s:.4f}' for s in fold_scores]}")
	logging.info(f"Average Model Performance: {mean_score:.4f} ± {std_score:.4f}")


Loading cropped images: 100%|██████████| 157/157 [00:28<00:00,  5.59it/s]


In [ ]:
main_training_pipeline()
